# 10A — Application readiness and data loading

## SAME STOPWATCH. DIFFERENT MOVEMENT. BETTER SCOUTING QUESTIONS.

**Purpose.** Milestone 10 builds the final product on the Milestone 09 application mart — it adds no warehouse scan, no model and no mart table. This notebook proves the mart can carry the final application: it re-runs the 09E integrity checks, runs the Milestone 10 logic checks (synthetic, known answers), checks the columns every new view reads, runs a few targeted integrity checks (stored twin caliper, 10 Hz sampling, phase mapping) and counts the no-data states the interface must handle.

**Prerequisites.** Notebook 09E executed in the same `RUN_MODE` (mart `warehouse/gold/advanced_movement/intelligence_v1/app_mart/v1`).

**Run.** Restart the kernel → run all. About a minute. Smoke mode reads the smoke mart and writes under `…/_sample/product_v1`.

**Writes.** `reports/tables/advanced_movement/product_v1/app_readiness_checks.csv`, `warehouse/gold/advanced_movement/product_v1/app_readiness.json`.

In [1]:
RUN_MODE = 'full'             # 'smoke' | 'full'
SEED = 20261011
FORCE_REBUILD = False
SAVE_TABLES = True
SAVE_FIGURES = True
NUM_WORKERS = 0

In [2]:
from nflbdb.data.config import Phase01Config, configure_runtime

cfg = Phase01Config.load()
configure_runtime(cfg)

import polars as pl
from IPython.display import Markdown, display

from nflbdb.data.utils import utc_now
from nflbdb.research.advanced_movement import config as amc
from nflbdb.research.advanced_movement import milestone09_artifacts as m9
from nflbdb.research.advanced_movement import milestone10_validation as v10
from nflbdb.research.advanced_movement import product_kit as pk
from nflbdb.research.advanced_movement.artifacts import ProductStore

controls = amc.NotebookControls(run_mode=RUN_MODE, force_rebuild=FORCE_REBUILD, save_figures=SAVE_FIGURES, save_tables=SAVE_TABLES,
                                seed=SEED, num_workers=NUM_WORKERS).validate()
store = ProductStore(cfg, controls)
MART = m9.mart_path(store)
T, EVID, MANIFEST = m9.load_mart(MART)
label = 'SMOKE — wiring check, numbers are not results' if controls.smoke else 'FULL'
display(Markdown(f'**Run mode: `{controls.run_mode}`** ({label}) · mart `{cfg.paths.rel(MART)}` built {MANIFEST["built_at"]} · {len(T)} tables'))

**Run mode: `full`** (FULL) · mart `warehouse/gold/advanced_movement/intelligence_v1/app_mart/v1` built 2026-10-14T13:13:05Z · 23 tables

## 1 · Mart integrity (the 09E acceptance checks, re-run on the files as they are now)

Version, required columns, row counts and fingerprints against the manifest, every displayed athlete in the profile table, every case a twin pair with three views, no grade column, abstentions without intervals. A changed file since 09E fails `fingerprints_match_manifest`: rerun 09E rather than editing the mart.

In [3]:
mart_report = m9.validate_mart(MART)
display(mart_report)
m9.assert_valid(mart_report)
print(f'{mart_report.height} mart checks passed')

check,passed,detail
str,bool,str
"""declared_version""",true,"""manifest 'v1', expected 'v1'"""
"""tables_present""",true,"""all present"""
"""required_columns""",true,"""all present"""
"""rows_match_manifest""",true,"""row counts equal the manifest"""
"""fingerprints_match_manifest""",true,"""files unchanged since the mani…"
…,…,…
"""no_grade_or_ranking_column""",true,"""no grade, potential or overall…"
"""similarity_labelled_descriptiv…",true,"""every similarity row carries i…"
"""unsupported_comparisons_abstai…",true,"""3453 abstention row(s) carry n…"


15 mart checks passed


## 2 · Milestone 10 logic (synthetic fixtures with known answers)

Phases never invented · replay never extrapolates · evidence labels · coverage classes · gap focus rule · comparison flags · simulator sections and categories · wording guard · story selection rules and word limits · token rendering. The same checks run in `tests/test_milestone10_product.py`.

In [4]:
logic_report = v10.run_all()
display(logic_report)
v10.assert_all_passed(logic_report)

check,status,detail
str,str,str
"""phase_decomposition_never_inve…","""PASS""","""absent phase stays 'not identi…"
"""replay_never_extrapolates""","""PASS""","""grid = recorded frames at 10 H…"
"""evidence_labels_and_stability""","""PASS""","""missing evidence is stated; la…"
"""coverage_classes_follow_the_re…","""PASS""","""classes derive from the analog…"
"""gap_summary_and_focus_rule""","""PASS""","""unsupported cells excluded fro…"
…,…,…
"""simulator_sections_and_categor…","""PASS""","""six structured sections; categ…"
"""simulator_audiences_and_wordin…","""PASS""","""three audiences render; no gra…"
"""wr_selection_rules""","""PASS""","""preferred pair only when it pa…"


## 3 · App-facing contracts

Every column a Milestone 10 view reads, by table. 09E guarantees a minimal set; the views also read optional columns of the same tables, so they are checked here before anyone opens the app.

In [5]:
APP_CONTRACT = {
    'stopwatch_twins': ['pair_id', 'position_group', 'nfl_id_a', 'nfl_id_b', 'name_a', 'name_b', 'draft_year_a', 'draft_year_b', 'forty_a', 'forty_b', 'd_forty',
                        'ten_a', 'ten_b', 'd_ten', 'ten_matched', 'd_weight', 'tier_a', 'tier_b', 'fingerprint_distance', 'n_features_compared',
                        'n_distinguishable', 'max_abs_z', 'top_feature'],
    'twin_differences': ['pair_id', 'feature', 'label', 'unit', 'phase', 'value_a', 'value_b', 'difference', 'ci_lower', 'ci_upper', 'z', 'verdict', 'quicker',
                         'evidence_status'],
    'prospect_profiles': ['nfl_id', 'display_name', 'position_group', 'draft_year', 'official_forty', 'official_ten', 'weight', 'height', 'evidence_tier',
                          'evidence_reasons', 'n_tracked_attempts', 'speed_capacity_pct', 'top_speed_pct', 'speed_development_pct'],
    'prospect_metrics': ['nfl_id', 'position_group', 'feature', 'label', 'unit', 'value', 'n_attempts', 'attempt_min', 'attempt_max', 'standard_error',
                         'pct_position', 'n_reference'],
    'feature_evidence': ['feature', 'label', 'phase', 'icc_within_position', 'r2_official_times', 'repeatability_label', 'redundancy_label', 'evidence_status',
                         'evidence_source', 'n_athletes', 'n_repeat', 'football_value'],
    'nfl_movement_demands': ['position_group', 'demand_id', 'label', 'n_episodes', 'n_athletes', 'n_eligible_plays', 'episodes_per_100_plays', 'share_plays_with',
                             'median_duration_s', 'median_distance_yd', 'supported'],
    'nfl_demand_registry': ['demand_id', 'label', 'combine_analogue', 'comparability', 'primary_feature', 'unit', 'physical_definition', 'features'],
    'movement_matching_summary': ['position_group', 'demand_id', 'n_prospects'],
    'movement_similarity_matrix': ['nfl_id', 'demand_id', 'n_dimensions', 'mean_coverage', 'coverage_lo', 'coverage_hi', 'distance', 'state'],
    'combine_attempt_curves': ['nfl_id', 'attempt_order', 'frame', 't_s', 'dist_yd', 'speed', 'a_tan', 'boundary'],
    'scouting_cases': ['case_id', 'case_type', 'pair_id', 'nfl_id_a', 'nfl_id_b', 'name_a', 'name_b', 'selected_using_nfl_outcome'],
}
contract = pl.DataFrame([{'table': k, 'present': k in T, 'n_columns_read': len(v),
                          'missing': ', '.join(c for c in v if c not in (T[k].columns if k in T else []))} for k, v in APP_CONTRACT.items()])
display(contract)
assert contract['present'].all() and (contract['missing'] == '').all(), 'a column read by the application is missing: fix before launching'

table,present,n_columns_read,missing
str,bool,i64,str
"""stopwatch_twins""",true,23,""""""
"""twin_differences""",true,14,""""""
"""prospect_profiles""",true,14,""""""
"""prospect_metrics""",true,12,""""""
"""feature_evidence""",true,12,""""""
…,…,…,…
"""nfl_demand_registry""",true,8,""""""
"""movement_matching_summary""",true,3,""""""
"""movement_similarity_matrix""",true,8,""""""


## 4 · Targeted integrity checks (only what a final view could display wrongly)

In [6]:
tw, d, cv = T['stopwatch_twins'], T['twin_differences'], T['combine_attempt_curves']
dt = cv.sort('seq_id', 'frame').with_columns(pl.col('t_s').diff().over('seq_id').alias('dt'))['dt'].drop_nulls()
max_gap = float(tw['d_forty'].abs().max())
feats = set(d['feature'].unique().to_list())
dem_ids = set(T['nfl_movement_demands']['demand_id'].unique().to_list())
reg_ids = set(T['nfl_demand_registry']['demand_id'].to_list())
checks = [
    ('twins_within_stored_caliper', max_gap <= 0.03 + 1e-9, f'max |Δ forty| = {max_gap:.3f} s (stored rule 0.03 s; app filters may only narrow it)'),
    ('every_pair_has_differences', set(tw['pair_id'].to_list()) <= set(d['pair_id'].to_list()), 'each twin pair has its 09C difference rows'),
    ('ten_hz_sampling', abs(float(dt.median()) - 0.1) < 0.01, f'median frame interval {float(dt.median()):.3f} s → the time-mode replay grid IS the recorded frames'),
    ('every_feature_has_a_sprint_phase', feats <= set(pk.PHASE_OF), f'features {sorted(feats)}; unmapped {sorted(feats - set(pk.PHASE_OF))}'),
    ('every_demand_in_registry', dem_ids <= reg_ids, f'{len(dem_ids)} demand ids'),
    ('cases_not_selected_on_nfl_outcome', not bool(T['scouting_cases']['selected_using_nfl_outcome'].any()), 'story inputs are pre-draft'),
    ('evidence_claims_present', bool(EVID.get('evidence_claims_08b6')), f"{len(EVID.get('evidence_claims_08b6', []))} executed 08B.6 claims"),
]
integrity = pl.DataFrame([{'check': c, 'passed': bool(ok), 'detail': det} for c, ok, det in checks])
display(integrity)
assert integrity['passed'].all(), 'targeted integrity check failed'

check,passed,detail
str,bool,str
"""twins_within_stored_caliper""",true,"""max |Δ forty| = 0.030 s (store…"
"""every_pair_has_differences""",true,"""each twin pair has its 09C dif…"
"""ten_hz_sampling""",true,"""median frame interval 0.100 s …"
"""every_feature_has_a_sprint_pha…",true,"""features ['acc_2_4', 'acc_4_6'…"
"""every_demand_in_registry""",true,"""9 demand ids"""
"""cases_not_selected_on_nfl_outc…",true,"""story inputs are pre-draft"""
"""evidence_claims_present""",true,"""20 executed 08B.6 claims"""


## 5 · No-data states the interface must handle

Each count is a real situation in this mart. The application shows a declared message for each (`product_kit.NO_DATA`) — never an invented value.

In [7]:
P = T['prospect_profiles']
twinned = set(tw['nfl_id_a'].to_list()) | set(tw['nfl_id_b'].to_list())
tracked = P.filter(pl.col('n_tracked_attempts') > 0)
sim = T['movement_similarity_matrix']
nodata = pl.DataFrame([
    {'state': 'no_tracked_attempt', 'count': int((P['n_tracked_attempts'] == 0).sum()), 'unit': 'prospects'},
    {'state': 'missing_split', 'count': int(P['official_ten'].is_null().sum()), 'unit': 'prospects'},
    {'state': 'single_attempt', 'count': int((P['n_tracked_attempts'] == 1).sum()), 'unit': 'prospects'},
    {'state': 'low_confidence', 'count': int(P['evidence_tier'].is_in(['low', 'medium']).sum()), 'unit': 'prospects (tier low / medium)'},
    {'state': 'no_twin', 'count': int(tracked.filter(~pl.col('nfl_id').is_in(list(twinned))).height), 'unit': 'tracked prospects'},
    {'state': 'unsupported_demand', 'count': int((sim['state'] == 'unsupported_by_forty').sum()), 'unit': 'prospect × demand rows'},
    {'state': 'no_nfl_observation', 'count': int((~T['nfl_movement_demands']['supported']).sum()), 'unit': 'position × demand cells'},
    {'state': 'no_feature', 'count': int(d['value_a'].is_null().sum() + d['value_b'].is_null().sum()), 'unit': 'difference rows with a missing value'},
]).with_columns(pl.col('state').map_elements(lambda s: pk.NO_DATA[s], return_dtype=pl.String).alias('message shown'))
display(nodata)

state,count,unit,message shown
str,i64,str,str
"""no_tracked_attempt""",92,"""prospects""","""No tracked forty attempt is re…"
"""missing_split""",85,"""prospects""","""Official 10-yard split not rec…"
"""single_attempt""",43,"""prospects""","""One recorded attempt: repeat-a…"
"""low_confidence""",86,"""prospects (tier low / medium)""","""Recording evidence is below th…"
"""no_twin""",3,"""tracked prospects""","""No stopwatch twin under the st…"
"""unsupported_demand""",2550,"""prospect × demand rows""","""This NFL movement demand has n…"
"""no_nfl_observation""",1,"""position × demand cells""","""Too few observed NFL episodes …"
"""no_feature""",1104,"""difference rows with a missing…","""This feature was not recorded …"


In [8]:
all_checks = pl.concat([mart_report.select('check', pl.col('passed'), 'detail').with_columns(pl.lit('09E mart').alias('group')),
                        logic_report.select('check', (pl.col('status') == 'PASS').alias('passed'), 'detail').with_columns(pl.lit('M10 logic').alias('group')),
                        integrity.with_columns(pl.lit('targeted').alias('group'))])
if controls.save_tables:
    store.tables_dir.mkdir(parents=True, exist_ok=True)
    all_checks.write_csv(store.tables_dir / 'app_readiness_checks.csv')
    nodata.write_csv(store.tables_dir / 'app_no_data_states.csv')
    store.write_json('app_readiness', {'finished_at': utc_now(), 'mart': cfg.paths.rel(MART), 'mart_built_at': MANIFEST['built_at'],
                                       'n_checks': all_checks.height, 'all_passed': bool(all_checks['passed'].all()), 'engine_version': pk.VERSION,
                                       'no_data_states': nodata.select('state', 'count').to_dicts()})
print(f'{all_checks.height} checks, all passed: {bool(all_checks["passed"].all())}')

33 checks, all passed: True


## 6 · Launch the application (optional, manual — from your own terminal, not from this notebook)

```
pip install -r apps/combine_intelligence_lab/requirements.txt
cd apps/combine_intelligence_lab
streamlit run app.py
```

Then walk through `reports/milestone10/APP_BROWSER_CHECKLIST.md`. The application reads the same mart this notebook validated.

**In football terms.** If every check above passed, each number the Lab shows comes from an executed table, the twin comparisons use the stored 0.03 s rule, the replay plays the actual 10 Hz frames, and every missing measurement has a stated message instead of a filled-in value.

**Send back:** the three check tables (sections 1, 2, 4) if anything fails, and the no-data table.